# Track 13: Decision Intelligence with RAG
## Bridging Evidence Retrieval, Multi-Criteria Optimization, Quantitative Risk Scoring, and Cryptographic Auditability

### Overview
Traditional RAG pipelines stop at text generation: they retrieve documents, format context, and prompt an LLM to generate an answer. However, in mission-critical enterprise environments—such as credit facility underwriting, infrastructure procurement, pharmaceutical trial selection, and regulatory compliance—**retrieval is not a decision**.

Decision Intelligence (DI) is the engineering discipline that transforms probabilistic, unstructured evidence into provably sound, risk-bounded, economically optimal, and auditable outcomes.

```
       ┌──────────────────────┐
       │   Evidence Sources   │  (OKF Bundles, JEV Index, Vector DB)
       └──────────┬───────────┘
                  │ Semantic & Structured Retrieval
                  ▼
       ┌──────────────────────┐
       │  Extraction & Normal.│  (Pydantic Type Validation, Noul Guard)
       └──────────┬───────────┘
                  │ Quantitative Attributes & Evidence Provenance
                  ▼
       ┌──────────────────────┐
       │     Policy Gates     │  (Hard Compliance Boundary Checks)
       └──────────┬───────────┘
                  │ Admissible Candidates
                  ▼
       ┌──────────────────────┐
       │    MCDA & Ranking    │  (TOPSIS, WSM, AHP Weights)
       └──────────┬───────────┘
                  │ Relative Closeness & Multi-Objective Tradeoffs
                  ▼
       ┌──────────────────────┐
       │ Risk & Econ Modeling │  (Expected Loss, VaR, Utility, Knapsack)
       └──────────┬───────────┘
                  │ Risk-Adjusted Payoffs
                  ▼
       ┌──────────────────────┐
       │ Confidence Protocol  │  (Abstain / Escalate HITL / Decide)
       └──────────┬───────────┘
                  │
                  ▼
       ┌──────────────────────┐
       │ Tamper-Evident Audit │  (SHA-256 Provenance Ledger)
       └──────────────────────┘
```


### Setup and Imports
Configure environment, paths, and load our decision intelligence components alongside TypeSafe JEV primitives.


In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path(".").resolve().parent.parent if Path(".").resolve().name == "13-decision-intelligence" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common.decision_intelligence import (
    Alternative,
    AutomatedPolicyEngine,
    ConfidenceAbstentionProtocol,
    Criterion,
    CriterionType,
    DecisionAuditRecord,
    DecisionRAGOrchestrator,
    DecisionSensitivityAnalyzer,
    DecisionStatus,
    EconomicOptimizer,
    MCDAEngine,
    PolicyGate,
    RiskScoringEngine,
    render_decision_intelligence_diagrams,
)
from common.typesafe_jev import Noul, Score, Choice

print(f"Project root set to: {PROJECT_ROOT}")
print("Decision Intelligence & JEV modules loaded successfully.")


### Visualizing the Architecture: Information Retrieval vs Decision Making
Before examining mathematical models, let us visualize the separation of concerns.
Retrieval surfaces candidate claims and evidentiary citations; Decision Intelligence evaluates trade-offs, constraints, risk boundaries, and generates immutable audit records.


In [ ]:
diagram_paths = render_decision_intelligence_diagrams(PROJECT_ROOT / "assets" / "diagrams")
from IPython.display import Image, display
display(Image(filename=str(PROJECT_ROOT / "assets" / "diagrams" / "track13_decision_intelligence_pipeline.png")))


### Section 1: Multi-Criteria Decision Analysis (MCDA)
In enterprise decision problems, alternatives have competing dimensions:
- Performance vs. Operating Cost
- Throughput vs. Latency
- SLA Guarantees vs. Price

MCDA provides mathematical rigor to rank candidates without arbitrary heuristics.
1. **Weighted Sum Model (WSM)**:
   - For benefit criteria: $r_{ij} = \frac{x_{ij} - \min(x_j)}{\max(x_j) - \min(x_j)}$
   - For cost criteria: $r_{ij} = \frac{\max(x_j) - x_{ij}}{\max(x_j) - \min(x_j)}$
   - Final utility: $U_i = \sum_{j} w_j \cdot r_{ij}$

2. **TOPSIS (Technique for Order of Preference by Similarity to Ideal Solution)**:
   - Vector normalization: $r_{ij} = \frac{x_{ij}}{\sqrt{\sum_k x_{kj}^2}}$
   - Weighted normalized matrix: $v_{ij} = w_j \cdot r_{ij}$
   - Separation from positive ideal ($A^+$) and negative ideal ($A^-$): $S_i^+ = \sqrt{\sum (v_{ij} - v_j^+)^2}, S_i^- = \sqrt{\sum (v_{ij} - v_j^-)^2}$
   - Relative closeness: $C_i = \frac{S_i^-}{S_i^+ + S_i^-}$, where $C_i \in [0, 1]$. Higher is better.


In [ ]:
# 1. Define evaluation criteria for infrastructure selection
criteria = [
    Criterion(name="latency_p99_ms", weight=0.25, criterion_type=CriterionType.COST, unit="ms"),
    Criterion(name="throughput_qps", weight=0.20, criterion_type=CriterionType.BENEFIT, unit="qps"),
    Criterion(name="security_compliance_score", weight=0.25, criterion_type=CriterionType.BENEFIT, unit="0-100"),
    Criterion(name="annual_tco_usd", weight=0.20, criterion_type=CriterionType.COST, unit="USD"),
    Criterion(name="sla_uptime_percentage", weight=0.10, criterion_type=CriterionType.BENEFIT, unit="%"),
]

# 2. Inspect normalized weights
weights_dict = MCDAEngine.normalize_weights(criteria)
print("Normalized Criteria Weights:")
for c, w in weights_dict.items():
    print(f"  {c:30s}: {w:.3f}")


### Analytic Hierarchy Process (AHP) Consistency Verification
Stakeholder criteria weights can suffer from cognitive bias or logical contradictions.
AHP calculates the principal eigenvalue $\lambda_{max}$, the Consistency Index ($CI = \frac{\lambda_{max} - n}{n - 1}$), and the Consistency Ratio ($CR = \frac{CI}{RI}$).
If $CR \le 0.10$, the pairwise comparison matrix is mathematically proven to be consistent and free from contradictory weight cycles.


In [ ]:
# 3x3 Pairwise comparison matrix for Latency, Throughput, and Security
pairwise_matrix = [
    [1.0,  2.0,  1.0],   # Latency vs [Latency, Throughput, Security]
    [0.5,  1.0,  0.5],   # Throughput vs [Latency, Throughput, Security]
    [1.0,  2.0,  1.0],   # Security vs [Latency, Throughput, Security]
]
criteria_subset = ["latency_p99_ms", "throughput_qps", "security_compliance_score"]

ahp_result = MCDAEngine.analytic_hierarchy_process(pairwise_matrix, criteria_subset)
print(f"AHP Principal Eigenvalue (lambda_max): {ahp_result['lambda_max']:.4f}")
print(f"Consistency Index (CI)               : {ahp_result['consistency_index']:.4f}")
print(f"Consistency Ratio (CR)               : {ahp_result['consistency_ratio']:.4f}")
print(f"Is Weight Assignment Consistent?     : {ahp_result['is_consistent']} (CR <= 0.10)")


### Section 2: Quantitative Risk Scoring & Uncertainty Bounds
Probabilistic RAG pipelines introduce uncertainty: missing documents, ambiguous extractions, or unverified claims.
We model risk quantitatively:
1. **Expected Loss ($E[L]$)**:
   $$E[L] = \sum_{k} P_k \times \text{Impact}_k$$
2. **Uncertainty-Penalized Composite Risk**:
   If evidence backing candidate attributes is incomplete or has low confidence ($C < 1.0$), we penalize the candidate:
   $$R_{\text{composite}} = E[L] \times \left(1 + \lambda_{\text{penalty}} \times (1 - C)\right)$$
3. **Parametric Value at Risk ($\text{VaR}_{95}$)**:
   Estimates the maximum loss at a 95% confidence level under normal variance approximation.


In [ ]:
# Evaluate risk profile for two candidate architectures
sample_risks_stable = [
    {"name": "minor_downtime", "probability": 0.05, "impact": 8.0},
    {"name": "minor_latency_drift", "probability": 0.15, "impact": 3.0},
]

sample_risks_untested = [
    {"name": "critical_compliance_breach", "probability": 0.35, "impact": 9.5},
    {"name": "data_poisoning_vulnerability", "probability": 0.20, "impact": 8.0},
]

risk_stable = RiskScoringEngine.calculate_composite_risk(
    sample_risks_stable, evidence_confidence=0.95
)
risk_untested = RiskScoringEngine.calculate_composite_risk(
    sample_risks_untested, evidence_confidence=0.60
)

print(f"Stable Vendor Risk   : Expected Loss = {risk_stable['expected_loss']:.2f}, Composite = {risk_stable['penalized_risk_score']:.2f}, VaR(95%) = {risk_stable['value_at_risk_95']:.2f}")
print(f"Untested Vendor Risk : Expected Loss = {risk_untested['expected_loss']:.2f}, Composite = {risk_untested['penalized_risk_score']:.2f}, VaR(95%) = {risk_untested['value_at_risk_95']:.2f}")


### Section 3: Economic Optimization & Portfolio Allocation
Decision intelligence must optimize financial and resource constraints:
1. **Expected Monetary Value (EMV)**: Weighted sum across states of nature.
2. **Von Neumann-Morgenstern Utility**: Incorporating risk tolerance ($\alpha$). Risk-averse decision makers prefer sure payoffs over volatile gambles with identical expected value.
3. **0-1 Knapsack Resource Allocation**: Given a finite budget, dynamically allocate capital across candidates to maximize utility.


In [ ]:
# 0-1 Knapsack resource constrained allocation
subsystems = [
    {"id": "sys_cache", "name": "Distributed Vector Cache", "cost": 15000.0, "value": 35.0},
    {"id": "sys_rerank", "name": "Cross-Encoder Hardware Reranker", "cost": 25000.0, "value": 55.0},
    {"id": "sys_graph", "name": "GraphRAG Entity Traversal Cluster", "cost": 35000.0, "value": 60.0},
    {"id": "sys_audit", "name": "Immutable Cryptographic Audit Logger", "cost": 10000.0, "value": 45.0},
]
budget_limit = 50000.0

allocation = EconomicOptimizer.knapsack_portfolio_selection(
    subsystems, budget_capacity=budget_limit
)
print(f"Allocated Budget : ${allocation['total_cost']:,.2f} / ${budget_limit:,.2f} ({allocation['budget_utilization']:.1%})")
print(f"Achieved Utility : {allocation['total_value']:.1f}")
print(f"Selected Modules : {allocation['selected_ids']}")


### Section 4: Automated Policy Decision Trees & TypeSafe JEV Gates
Decision systems must enforce strict, non-negotiable policy constraints (e.g. SOC2 compliance, latency ceiling, SLA floor).
Crucially, when attributes are missing from retrieved documents, ordinary code often defaults to `0` or `None`, which can cause false passes or silent logic bugs.

We integrate **TypeSafe JEV** primitives:
- If a value is missing or unverified, it evaluates to `Noul(reason=...)`.
- The `PolicyGate` detects `Noul` and immediately halts or flags the alternative, preventing unsafe decisions.


In [ ]:
# Create deterministic policy gates
policy_gates = [
    PolicyGate(
        name="mandatory_security_compliance",
        criterion="security_compliance_score",
        operator=">=",
        threshold=80.0,
        rejection_action=DecisionStatus.REJECTED
    ),
    PolicyGate(
        name="max_acceptable_latency",
        criterion="latency_p99_ms",
        operator="<=",
        threshold=25.0,
        rejection_action=DecisionStatus.REJECTED
    ),
    PolicyGate(
        name="enterprise_sla_gate",
        criterion="sla_uptime_percentage",
        operator=">=",
        threshold=99.95,
        rejection_action=DecisionStatus.ESCALATED_HUMAN_REVIEW
    ),
]

policy_engine = AutomatedPolicyEngine(policy_gates)

# Test evaluation of candidate with missing SLA attribute (Noul sentinel)
test_candidate = Alternative(
    id="vendor_incomplete",
    name="Incomplete Spec Vendor",
    attributes={
        "security_compliance_score": 90.0,
        "latency_p99_ms": 12.0
        # Notice sla_uptime_percentage is omitted!
    }
)

eval_result = policy_engine.evaluate_alternative(test_candidate)
print(f"Is Admissible      : {eval_result['is_admissible']}")
print(f"Recommended Action : {eval_result['recommended_action'].value if hasattr(eval_result['recommended_action'], 'value') else eval_result['recommended_action']}")
print("Audit Details:")
for entry in eval_result['audit_trail']:
    print(f"  [{entry['gate_name']}]: Passed={entry['passed']} -> {entry['detail']}")


### Section 5: Confidence, Abstention, and Escalation Protocol
A trustworthy AI decision engine must know when **not** to make an automated decision:
1. **Abstention**: Triggered when evidence confidence from RAG retrieval is below safety thresholds ($C < \theta_{\text{abstain}}$).
2. **Escalation**: Triggered when the margin of victory between top candidates is smaller than the ambiguity threshold ($\Delta < \epsilon_{\text{margin}}$), or when composite risk exceeds acceptable risk appetite.
3. **Decided**: Clear winning candidate with verified evidentiary backing.


In [ ]:
# Test ambiguity escalation scenario: two vendors virtually tied
cand_alpha = Alternative(id="cand_alpha", name="Alpha High-End", confidence=0.96)
cand_beta = Alternative(id="cand_beta", name="Beta Budget", confidence=0.95)

rankings_tied = [
    {"alternative_id": "cand_alpha", "score": 0.812},
    {"alternative_id": "cand_beta", "score": 0.809},  # Difference is only 0.003!
]

status, rationale, winner_id = ConfidenceAbstentionProtocol.assess_decision(
    top_rankings=rankings_tied,
    admissible_alternatives=[cand_alpha, cand_beta],
    ambiguity_margin_threshold=0.03
)
print(f"Decision Status : {status.value}")
print(f"Rationale       : {rationale}")


### Section 6: End-to-End Case Study with Shared Fixture
We load our shared fixture (`fixtures/sample_decision_records/cloud_vendor_selection.json`) which models a real-world enterprise infrastructure procurement decision across four vendor alternatives.
The orchestrator:
1. Filters candidates through Policy Gates.
2. Executes TOPSIS and WSM ranking on admissible alternatives.
3. Performs risk scoring and economic utility modeling.
4. Checks confidence bounds and ambiguity margins.
5. Computes counterfactual sensitivity flips.
6. Emits an immutable SHA-256 tamper-evident audit record.


In [ ]:
fixture_file = PROJECT_ROOT / "fixtures" / "sample_decision_records" / "cloud_vendor_selection.json"
with open(fixture_file, "r", encoding="utf-8") as f:
    fixture_data = json.load(f)

fixture_criteria = [Criterion(**c) for c in fixture_data["criteria"]]
fixture_gates = [PolicyGate(**g) for g in fixture_data["policy_gates"]]
fixture_alts = [Alternative(**a) for a in fixture_data["alternatives"]]

orchestrator = DecisionRAGOrchestrator(
    criteria=fixture_criteria,
    policy_gates=fixture_gates,
    budget_constraint=fixture_data["budget_constraint_usd"]
)

audit_record = orchestrator.solve(
    objective=fixture_data["objective"],
    alternatives=fixture_alts,
    min_evidence_confidence=0.75,
    ambiguity_margin=0.03,
)

print(f"Decision ID        : {audit_record.decision_id}")
print(f"Decision Status    : {audit_record.status.value}")
print(f"Selected Winner    : {audit_record.selected_alternative_id}")
print(f"SHA-256 Fingerprint: {audit_record.provenance_hash}")


### Detailed Audit Trail & Counterfactual Sensitivity Flips
Let us inspect the audit records, gate rejections, and the counterfactual sensitivity analysis.
Notice how OpenVector is rejected by the mandatory security compliance gate (score 74 < 80 threshold), and how weight adjustments on latency or cost affect the top ranking.


In [ ]:
print("=== 1. TOPSIS & WSM Composite Rankings ===")
for r in audit_record.rankings:
    print(f"Rank {r['rank']}: {r['alternative_id']:15s} | TOPSIS Closeness: {r['score']:.4f}")

print("
=== 2. Policy Gate Admissibility Status ===")
for entry in audit_record.policy_audit_trail:
    act = entry['recommended_action'].value if hasattr(entry['recommended_action'], 'value') else entry['recommended_action']
    print(f"Alternative: {entry['alternative_id']:15s} | Admissible: {entry['is_admissible']} | Action: {act}")

print("
=== 3. Counterfactual Sensitivity Analysis ===")
sens = audit_record.counterfactual_sensitivity
print(f"Base Winner: {sens.get('base_winner')}")
print(f"Robustness Score: {sens.get('robustness_score', 0.0):.2%}")
if sens.get("flips"):
    print("Potential Ranking Flips:")
    for flip in sens["flips"]:
        print(f"  Criterion '{flip['criterion']}' {flip['direction']} by {abs(flip['delta_weight']):.2f} -> Winner becomes '{flip['new_winner']}'")


### Visualizing MCDA Vendor Relative Closeness
Let us view the generated visual comparison of TOPSIS relative closeness across all evaluated alternatives.


In [ ]:
display(Image(filename=str(PROJECT_ROOT / "assets" / "diagrams" / "track13_mcda_vendor_comparison.png")))


### Summary and Pedagogical Takeaways
In this track, we moved beyond basic retrieval and LLM prompting to construct a complete Decision Intelligence system:
1. **Information Retrieval vs Decision Making**: Retrieval extracts candidate claims; Decision Intelligence evaluates tradeoffs, enforces compliance, models risks, and optimizes objectives.
2. **Multi-Criteria Decision Analysis (MCDA)**: Implemented TOPSIS and Weighted Sum Model with cost/benefit normalization, and validated pairwise transitivity using Analytic Hierarchy Process (AHP).
3. **Quantitative Risk Models**: Incorporated Expected Loss and Value at Risk (VaR), penalizing candidates with uncertain evidentiary backing.
4. **TypeSafe Policy Gates**: Prevented silent defaults and hallucinations by treating unverified or missing attributes as `Noul` sentinels.
5. **Confidence, Abstention & Escalation**: Designed safety protocols to abstain on uncertain retrieval and escalate to human review when alternatives are ambiguously close.
6. **Immutable Provenance Audit Trail**: Built tamper-evident SHA-256 hash chains and counterfactual sensitivity models to prove why an alternative was chosen.

Next, in **Track 14 — Enterprise RAG**, we explore asynchronous ingestion, multi-tenancy ACL enforcement, scale, and operational production architectures.
